[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/067.%20The%20Kitting%20%26%20Value-Added%20Service%20Scheduling%20Problem/P67-Tier-1_executed.ipynb)

---


# 67. The Kitting & Value-Added Service Scheduling Problem
## Tier 1: The Pen-and-Paper Method (Mathematical Formulation)

## Purpose and scope

This notebook formulates and solves a deterministic kitting schedule with a mixed-integer linear programming (MILP) model. The model chooses how many units of each kit to produce in each shift and period. It balances production labor cost, finished-kit inventory cost, component-SKU inventory cost, and shortage penalties, subject to shift capacity, inventory-balance, and finished-kit storage constraints.

The inputs are kit demand by period, each kit's processing time, the bill of materials, labor and inventory costs, shift capacities, component supply, and storage capacity. The outputs are a production schedule, ending inventories, shortages, a cost breakdown, capacity utilization, and sensitivity results for the specified capacity scenarios.

### Model assumptions

- Demand is known and deterministic for each kit and period.
- Each period has the same indexed shifts; production in a period can satisfy demand in that period or later periods through finished-kit inventory.
- Initial finished-kit and component-SKU inventories are zero.
- Each SKU receives 500 units in every period. Component supply is fixed and is not a decision variable; component shortages are not allowed.
- Kit production consumes components according to the bill of materials. Component inventory is carried between periods and incurs a holding cost.
- Processing time is measured in minutes per kit. Shift capacity is measured in minutes, and labor cost is measured in dollars per hour.
- Finished-kit inventory is limited by a shared storage capacity at the end of each period. The model does not impose a storage constraint on component SKUs.
- Shortages are allowed and penalized. The model has no setup cost, purchase cost, overtime decision, or terminal-inventory requirement.
- Production quantities are integer. Inventory and shortage variables are continuous and nonnegative.

## Method and teaching sequence

The solution follows the mathematical-formulation approach: define decision variables, construct the total-cost objective, add the inventory and capacity constraints, solve the MILP with PuLP's CBC solver, and interpret the resulting schedule and cost components. CBC is an exact MILP solver: when it reports an optimal status, the reported objective is optimal for this model and these input data, subject to the solver's numerical tolerances.

The example uses 2 SKUs, 2 kit types, 2 shifts, and 3 periods:

- Kit 1 requires 2 units of SKU 1 and 1 unit of SKU 2.
- Kit 2 requires 1 unit of SKU 1 and 3 units of SKU 2.
- Processing times are 3 minutes per Kit 1 and 4 minutes per Kit 2.
- Each period has shift capacities of 480 minutes for Shift 1 and 420 minutes for Shift 2.

The data and capacity scenarios are the inputs to the model; results are produced by running the notebook. Existing outputs are not used as evidence of a new run.

## Mathematical model

### Indices, parameters, and decisions

Let $k$ index kit types, $i$ index component SKUs, $s$ index shifts, and $t$ index planning periods. The code uses zero-based Python indices; the mathematical notation uses the same sets without specifying their starting index.

The parameters are:

- $d_{kt}$: demand for kit $k$ in period $t$ (`problem.demand[k, t]`).
- $p_k$: processing time per unit of kit $k$, in minutes (`problem.processing_time[k]`).
- $a_{ik}$: units of SKU $i$ needed for one unit of kit $k$ (`problem.bill_of_materials[i, k]`).
- $c_{st}$: labor cost per hour in shift $s$ and period $t$ (`problem.labor_cost[s, t]`).
- $h^{K}_{kt}$ and $h^{S}_{it}$: holding costs per unit of kit and SKU (`problem.holding_cost_kit[k, t]` and `problem.holding_cost_sku[i, t]`).
- $C_{st}$: available minutes in shift $s$ and period $t$ (`problem.capacity[s, t]`).
- $q_{it}$: supply of SKU $i$ in period $t$ (`problem.sku_supply[i, t]`).
- $H$: finished-kit storage capacity (`problem.storage_capacity`), and $\pi$: shortage penalty per kit (`problem.shortage_penalty`).

The decision variables are:

- $x_{kst}$: integer number of kits of type $k$ produced in shift $s$, period $t$ (`x[(k, s, t)]`).
- $y_{kst}$: binary production-link variable (`y[(k, s, t)]`). The linking constraint requires $y_{kst}=1$ whenever $x_{kst}>0$. There is no setup cost or other constraint that rewards or requires $y_{kst}=0$ when production is zero, so this variable is a link indicator rather than a priced setup decision.
- $I^{K}_{kt}$ and $I^{S}_{it}$: nonnegative ending inventories of kits and SKUs (`inv_kit[(k, t)]` and `inv_sku[(i, t)]`).
- $r_{kt}$: nonnegative shortage of kit $k$ in period $t$ (`short[(k, t)]`).

### Objective: minimize total cost

The objective adds production labor cost, finished-kit holding cost, component-SKU holding cost, and shortage penalties:

$$
\begin{aligned}
\min \quad
& \sum_{k,s,t} \frac{c_{st}}{60}p_k x_{kst}
+ \sum_{k,t} h^{K}_{kt} I^{K}_{kt}
+ \sum_{i,t} h^{S}_{it} I^{S}_{it}
+ \pi \sum_{k,t} r_{kt}.
\end{aligned}
$$

The factor $1/60$ converts the processing time from minutes to hours so it is consistent with labor cost in dollars per hour. In the code, these four terms are `production_cost`, `inventory_cost_kit`, `inventory_cost_sku`, and `shortage_cost`.

### Inventory balances and capacity constraints

A balance equation accounts for what is available, what is produced or consumed, what is demanded, and what remains at the end of the period. Shortage appears as a nonnegative amount that reduces the demand that must be met from available kits. With zero initial inventory, the kit balance is:

$$
I^{K}_{k,t-1} + \sum_s x_{kst} - d_{kt} + r_{kt} = I^{K}_{kt},
\qquad I^{K}_{k,0^-}=0.
$$

For each component SKU, production consumes the bill-of-materials quantity and the period's supply is added to inventory:

$$
I^{S}_{i,t-1} + q_{it} - \sum_{k,s} a_{ik}x_{kst} = I^{S}_{it},
\qquad I^{S}_{i,0^-}=0.
$$

The remaining constraints are:

$$
\begin{aligned}
\sum_k p_k x_{kst} &\le C_{st} && \text{for each shift and period},\\
\sum_k I^{K}_{kt} &\le H && \text{for each period},\\
x_{kst} &\le M y_{kst} && \text{for each kit, shift, and period},\\
x_{kst} &\in \mathbb{Z}_{\ge 0},\quad y_{kst}\in\{0,1\},\\
I^{K}_{kt},\ I^{S}_{it},\ r_{kt} &\ge 0.
\end{aligned}
$$

Here $M$ is the linking constant (`M = 10000`). The first constraint limits processing minutes in a shift; the second limits total finished-kit inventory, not component inventory. The link ensures that positive production requires the corresponding binary variable to be active. The model is a MILP because it includes integer and binary decisions alongside continuous inventory and shortage variables. The number of variables and constraints grows with the number of kits, SKUs, shifts, and periods; general MILPs can require substantial computation as these dimensions grow.

The solver's objective is the minimum total cost over schedules satisfying all these constraints. The later analysis decomposes that objective and reports service level and capacity use so the solution can be interpreted operationally.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pulp

from dataclasses import dataclass
from typing import Dict, Tuple

# Consistent plotting defaults make the schedules and cost components easier to compare.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 1. Define the problem data

The data class gathers the dimensions and numerical parameters used by the model. Array shapes follow the index order shown in the mathematical model: for example, demand has shape `(kit, period)`, capacity has shape `(shift, period)`, and the bill of materials has shape `(SKU, kit)`. Keeping these relationships explicit helps prevent a value from being applied to the wrong decision or constraint.

In [ ]:
@dataclass
class KittingProblem:
    """Input data for the deterministic kitting scheduling MILP."""

    num_skus: int
    num_kits: int
    num_shifts: int
    num_periods: int

    demand: np.ndarray  # [kit, period]
    processing_time: np.ndarray  # [kit], minutes per kit
    holding_cost_sku: np.ndarray  # [SKU, period], dollars per unit
    holding_cost_kit: np.ndarray  # [kit, period], dollars per unit
    labor_cost: np.ndarray  # [shift, period], dollars per hour
    capacity: np.ndarray  # [shift, period], minutes
    bill_of_materials: np.ndarray  # [SKU, kit], units of SKU per kit
    sku_supply: np.ndarray  # [SKU, period], units supplied in each period
    shortage_penalty: float = 10.0
    storage_capacity: float = 1000.0


def create_example_problem() -> KittingProblem:
    """Create the deterministic example used throughout this notebook."""
    return KittingProblem(
        num_skus=2,
        num_kits=2,
        num_shifts=2,
        num_periods=3,
        demand=np.array([
            [100, 90, 80],  # Kit 1 demand by period
            [50, 60, 40],   # Kit 2 demand by period
        ]),
        processing_time=np.array([3, 4]),
        holding_cost_sku=np.array([
            [0.50, 0.50, 0.50],  # SKU 1
            [0.50, 0.50, 0.50],  # SKU 2
        ]),
        holding_cost_kit=np.array([
            [0.80, 0.80, 0.80],  # Kit 1
            [1.00, 1.00, 1.00],  # Kit 2
        ]),
        labor_cost=np.array([
            [15.0, 15.0, 15.0],  # Shift 1, dollars per hour
            [18.0, 18.0, 18.0],  # Shift 2, dollars per hour
        ]),
        capacity=np.array([
            [480, 480, 480],  # Shift 1, minutes per period
            [420, 420, 420],  # Shift 2, minutes per period
        ]),
        bill_of_materials=np.array([
            [2, 1],  # SKU 1: units required by Kit 1 and Kit 2
            [1, 3],  # SKU 2: units required by Kit 1 and Kit 2
        ]),
        sku_supply=np.full((2, 3), 500),
        shortage_penalty=10.0,
        storage_capacity=1000.0,
    )


problem = create_example_problem()
print(
    f"Problem created: {problem.num_kits} kits, {problem.num_skus} SKUs, "
    f"{problem.num_shifts} shifts, {problem.num_periods} periods"
)

## 2. Formulate and solve the MILP

The solver function builds the model in the same order as the formulation: variables, objective, inventory balances, capacity and storage limits, then production-link constraints. The kit and SKU balances carry inventory from one period to the next, so production and supply in an early period can affect later periods. CBC returns the status and objective; the function extracts the decision-variable values only when the model is optimal, so a failed or non-optimal solve is not silently presented as a usable schedule.

In [ ]:
def solve_kitting_mip(problem: KittingProblem) -> Tuple[Dict, pulp.LpProblem]:
    """Build and solve the kitting scheduling model with CBC."""
    model = pulp.LpProblem("Kitting_Scheduling", pulp.LpMinimize)

    # Integer production and binary link variables represent the scheduling decisions.
    x = {
        (k, s, t): pulp.LpVariable(
            f"x_{k}_{s}_{t}", 0, cat="Integer"
        )
        for k in range(problem.num_kits)
        for s in range(problem.num_shifts)
        for t in range(problem.num_periods)
    }
    y = {
        (k, s, t): pulp.LpVariable(
            f"y_{k}_{s}_{t}", cat="Binary"
        )
        for k in range(problem.num_kits)
        for s in range(problem.num_shifts)
        for t in range(problem.num_periods)
    }
    inv_kit = {
        (k, t): pulp.LpVariable(
            f"inv_kit_{k}_{t}", 0, cat="Continuous"
        )
        for k in range(problem.num_kits)
        for t in range(problem.num_periods)
    }
    inv_sku = {
        (i, t): pulp.LpVariable(
            f"inv_sku_{i}_{t}", 0, cat="Continuous"
        )
        for i in range(problem.num_skus)
        for t in range(problem.num_periods)
    }
    short = {
        (k, t): pulp.LpVariable(
            f"short_{k}_{t}", 0, cat="Continuous"
        )
        for k in range(problem.num_kits)
        for t in range(problem.num_periods)
    }

    production_cost = pulp.lpSum(
        (problem.labor_cost[s, t] / 60)
        * problem.processing_time[k]
        * x[(k, s, t)]
        for k in range(problem.num_kits)
        for s in range(problem.num_shifts)
        for t in range(problem.num_periods)
    )
    inventory_cost_kit = pulp.lpSum(
        problem.holding_cost_kit[k, t] * inv_kit[(k, t)]
        for k in range(problem.num_kits)
        for t in range(problem.num_periods)
    )
    inventory_cost_sku = pulp.lpSum(
        problem.holding_cost_sku[i, t] * inv_sku[(i, t)]
        for i in range(problem.num_skus)
        for t in range(problem.num_periods)
    )
    shortage_cost = pulp.lpSum(
        problem.shortage_penalty * short[(k, t)]
        for k in range(problem.num_kits)
        for t in range(problem.num_periods)
    )

    model += (
        production_cost
        + inventory_cost_kit
        + inventory_cost_sku
        + shortage_cost
    )

    # The first-period balances use zero initial inventory; later balances carry inventory forward.
    for k in range(problem.num_kits):
        for t in range(problem.num_periods):
            prior_inventory = 0 if t == 0 else inv_kit[(k, t - 1)]
            model += (
                prior_inventory
                + pulp.lpSum(
                    x[(k, s, t)] for s in range(problem.num_shifts)
                )
                - problem.demand[k, t]
                + short[(k, t)]
                == inv_kit[(k, t)]
            )

    for i in range(problem.num_skus):
        for t in range(problem.num_periods):
            prior_inventory = 0 if t == 0 else inv_sku[(i, t - 1)]
            sku_consumption = pulp.lpSum(
                problem.bill_of_materials[i, k] * x[(k, s, t)]
                for k in range(problem.num_kits)
                for s in range(problem.num_shifts)
            )
            model += (
                prior_inventory
                + problem.sku_supply[i, t]
                - sku_consumption
                == inv_sku[(i, t)]
            )

    for s in range(problem.num_shifts):
        for t in range(problem.num_periods):
            model += pulp.lpSum(
                problem.processing_time[k] * x[(k, s, t)]
                for k in range(problem.num_kits)
            ) <= problem.capacity[s, t]

    for t in range(problem.num_periods):
        model += pulp.lpSum(
            inv_kit[(k, t)] for k in range(problem.num_kits)
        ) <= problem.storage_capacity

    # This big-M link makes the binary variable active whenever production is positive.
    big_m = 10000
    for k in range(problem.num_kits):
        for s in range(problem.num_shifts):
            for t in range(problem.num_periods):
                model += x[(k, s, t)] <= big_m * y[(k, s, t)]

    print("Solving the kitting MILP with CBC...")
    model.solve(pulp.PULP_CBC_CMD(msg=False))
    status = pulp.LpStatus[model.status]

    if status != "Optimal":
        raise RuntimeError(f"The MILP did not reach optimality. Solver status: {status}")

    objective = pulp.value(model.objective)
    if objective is None:
        raise RuntimeError("The solver returned no objective value.")

    results = {
        "status": status,
        "objective": float(objective),
        "production_schedule": np.zeros(
            (problem.num_kits, problem.num_shifts, problem.num_periods)
        ),
        "kit_inventory": np.zeros((problem.num_kits, problem.num_periods)),
        "sku_inventory": np.zeros((problem.num_skus, problem.num_periods)),
        "shortages": np.zeros((problem.num_kits, problem.num_periods)),
    }

    for k in range(problem.num_kits):
        for s in range(problem.num_shifts):
            for t in range(problem.num_periods):
                results["production_schedule"][k, s, t] = x[(k, s, t)].varValue

    for k in range(problem.num_kits):
        for t in range(problem.num_periods):
            results["kit_inventory"][k, t] = inv_kit[(k, t)].varValue
            results["shortages"][k, t] = short[(k, t)].varValue

    for i in range(problem.num_skus):
        for t in range(problem.num_periods):
            results["sku_inventory"][i, t] = inv_sku[(i, t)].varValue

    return results, model


results, model = solve_kitting_mip(problem)
print(f"Solution status: {results['status']}")
print(f"Total cost: ${results['objective']:.2f}")

## 3. Analyze the schedule and cost

The production schedule reports kit quantities by shift and period. The cost breakdown recomputes each objective component from the extracted decisions, then checks that their sum agrees with the solver objective. Capacity utilization compares processing minutes used with minutes available. The service-level measure is demand-weighted across kits: total demand minus total shortage, divided by total demand. A shortage is unmet demand in this model; it is penalized, not prohibited.

In [ ]:
def analyze_solution(results: Dict, problem: KittingProblem) -> None:
    """Report the schedule, objective components, capacity, and service level."""
    print("=" * 60)
    print("OPTIMAL KITTING SCHEDULE ANALYSIS")
    print("=" * 60)

    print("\n1. PRODUCTION SCHEDULE (units per shift per period):")
    print("-" * 50)
    for k in range(problem.num_kits):
        print(f"\nKit {k + 1}:")
        schedule_df = pd.DataFrame(
            results["production_schedule"][k],
            index=[f"Shift {s + 1}" for s in range(problem.num_shifts)],
            columns=[f"Period {t + 1}" for t in range(problem.num_periods)],
        )
        print(schedule_df.round(1))
        total_per_period = results["production_schedule"][k].sum(axis=0)
        formatted_totals = ", ".join(f"{value:.1f}" for value in total_per_period)
        print(f"Total production by period: [{formatted_totals}]")

    production_cost = sum(
        (problem.labor_cost[s, t] / 60)
        * problem.processing_time[k]
        * results["production_schedule"][k, s, t]
        for k in range(problem.num_kits)
        for s in range(problem.num_shifts)
        for t in range(problem.num_periods)
    )
    inventory_cost_kit = float(
        np.sum(problem.holding_cost_kit * results["kit_inventory"])
    )
    inventory_cost_sku = float(
        np.sum(problem.holding_cost_sku * results["sku_inventory"])
    )
    shortage_cost = float(
        problem.shortage_penalty * np.sum(results["shortages"])
    )
    component_total = (
        production_cost
        + inventory_cost_kit
        + inventory_cost_sku
        + shortage_cost
    )
    assert np.isclose(component_total, results["objective"], rtol=1e-7, atol=1e-6)

    print("\n2. COST BREAKDOWN:")
    print("-" * 30)
    print(f"Production cost: ${production_cost:.2f}")
    print(f"Kit inventory cost: ${inventory_cost_kit:.2f}")
    print(f"SKU inventory cost: ${inventory_cost_sku:.2f}")
    print(f"Shortage cost: ${shortage_cost:.2f}")
    print(f"Total cost: ${results['objective']:.2f}")

    print("\n3. CAPACITY UTILIZATION:")
    print("-" * 30)
    for s in range(problem.num_shifts):
        print(f"\nShift {s + 1}:")
        for t in range(problem.num_periods):
            used_capacity = sum(
                problem.processing_time[k]
                * results["production_schedule"][k, s, t]
                for k in range(problem.num_kits)
            )
            utilization = 100 * used_capacity / problem.capacity[s, t]
            print(
                f"  Period {t + 1}: {used_capacity:.1f}/"
                f"{problem.capacity[s, t]} min ({utilization:.1f}%)"
            )

    total_demand = float(np.sum(problem.demand))
    total_shortage = float(np.sum(results["shortages"]))
    service_level = 100 * (total_demand - total_shortage) / total_demand

    print("\n4. SERVICE LEVEL ANALYSIS:")
    print("-" * 30)
    print(f"Total demand: {total_demand:.1f} units")
    print(f"Total shortage: {total_shortage:.1f} units")
    print(f"Demand-weighted service level: {service_level:.1f}%")

    for k in range(problem.num_kits):
        kit_demand = float(np.sum(problem.demand[k]))
        kit_shortage = float(np.sum(results["shortages"][k]))
        kit_service = 100 * (kit_demand - kit_shortage) / kit_demand
        print(f"  Kit {k + 1}: {kit_service:.1f}% service level")


analyze_solution(results, problem)

## 4. Visualize the solution

The plots provide complementary views of the same schedule: total kit units produced by shift and period, the four objective components, capacity utilization, and ending finished-kit inventory over time. Because the production heatmap adds units of different kit types, it is a unit count rather than a processing-time or cost measure; capacity utilization is shown separately using processing minutes.

In [ ]:
def create_comprehensive_visualization(results: Dict, problem: KittingProblem):
    """Plot production, objective components, utilization, and kit inventory."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "Kitting and Value-Added Service Scheduling Results",
        fontsize=16,
        fontweight="bold",
    )

    total_production = results["production_schedule"].sum(axis=0)
    sns.heatmap(
        total_production,
        annot=True,
        fmt=".1f",
        cmap="YlOrRd",
        xticklabels=[f"Period {t + 1}" for t in range(problem.num_periods)],
        yticklabels=[f"Shift {s + 1}" for s in range(problem.num_shifts)],
        cbar_kws={"label": "Kit units produced"},
        ax=axes[0, 0],
    )
    axes[0, 0].set_title("Production Schedule (All Kit Types)")
    axes[0, 0].set_xlabel("Planning period")
    axes[0, 0].set_ylabel("Shift")

    production_cost = sum(
        (problem.labor_cost[s, t] / 60)
        * problem.processing_time[k]
        * results["production_schedule"][k, s, t]
        for k in range(problem.num_kits)
        for s in range(problem.num_shifts)
        for t in range(problem.num_periods)
    )
    inventory_cost_kit = float(
        np.sum(problem.holding_cost_kit * results["kit_inventory"])
    )
    inventory_cost_sku = float(
        np.sum(problem.holding_cost_sku * results["sku_inventory"])
    )
    shortage_cost = float(
        problem.shortage_penalty * np.sum(results["shortages"])
    )
    cost_names = ["Production", "Kit inventory", "SKU inventory", "Shortage"]
    cost_values = [
        production_cost,
        inventory_cost_kit,
        inventory_cost_sku,
        shortage_cost,
    ]
    bars = axes[0, 1].bar(
        cost_names,
        cost_values,
        color=["#ff7f0e", "#2ca02c", "#d62728", "#9467bd"],
    )
    axes[0, 1].set_title("Objective Cost Components")
    axes[0, 1].set_ylabel("Cost (dollars)")
    axes[0, 1].tick_params(axis="x", rotation=25)
    for bar, value in zip(bars, cost_values):
        axes[0, 1].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"${value:.0f}",
            ha="center",
            va="bottom",
        )

    periods = np.arange(problem.num_periods)
    width = 0.8 / problem.num_shifts
    for s in range(problem.num_shifts):
        utilizations = []
        for t in range(problem.num_periods):
            used_capacity = sum(
                problem.processing_time[k]
                * results["production_schedule"][k, s, t]
                for k in range(problem.num_kits)
            )
            utilizations.append(100 * used_capacity / problem.capacity[s, t])
        axes[1, 0].bar(
            periods + width * s,
            utilizations,
            width,
            label=f"Shift {s + 1}",
        )
    axes[1, 0].set_title("Capacity Utilization by Shift and Period")
    axes[1, 0].set_xlabel("Planning period")
    axes[1, 0].set_ylabel("Capacity used (%)")
    axes[1, 0].set_xticks(
        periods + width * (problem.num_shifts - 1) / 2,
        [f"Period {t + 1}" for t in range(problem.num_periods)],
    )
    axes[1, 0].legend(title="Shift")
    axes[1, 0].grid(True, axis="y", alpha=0.3)

    period_numbers = np.arange(1, problem.num_periods + 1)
    for k in range(problem.num_kits):
        axes[1, 1].plot(
            period_numbers,
            results["kit_inventory"][k],
            marker="o",
            linewidth=2,
            label=f"Kit {k + 1}",
        )
    axes[1, 1].set_title("Ending Kit Inventory by Period")
    axes[1, 1].set_xlabel("Planning period")
    axes[1, 1].set_ylabel("Inventory (kit units)")
    axes[1, 1].set_xticks(period_numbers)
    axes[1, 1].legend(title="Kit type")
    axes[1, 1].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
    return fig


fig = create_comprehensive_visualization(results, problem)

## 5. Capacity sensitivity analysis

Sensitivity analysis asks how the optimized cost, service level, and total production respond when a model parameter changes. The experiment keeps demand, costs, supply, and all other assumptions fixed, and scales every shift-period capacity by the specified factor. Each scenario is built as a separate problem instance, so the baseline `problem` and its solved results remain unchanged. These scenario solves use the same MILP formulation and solver as the main example; they are not a different solution method.

In [ ]:
def sensitivity_analysis(problem: KittingProblem) -> list:
    """Compare the MILP solution across the specified capacity factors."""
    print("=" * 60)
    print("CAPACITY SENSITIVITY ANALYSIS")
    print("=" * 60)

    capacity_scenarios = [0.8, 1.0, 1.2]
    results_comparison = []

    for scenario in capacity_scenarios:
        modified_problem = KittingProblem(
            num_skus=problem.num_skus,
            num_kits=problem.num_kits,
            num_shifts=problem.num_shifts,
            num_periods=problem.num_periods,
            demand=problem.demand.copy(),
            processing_time=problem.processing_time.copy(),
            holding_cost_sku=problem.holding_cost_sku.copy(),
            holding_cost_kit=problem.holding_cost_kit.copy(),
            labor_cost=problem.labor_cost.copy(),
            capacity=problem.capacity * scenario,
            bill_of_materials=problem.bill_of_materials.copy(),
            sku_supply=problem.sku_supply.copy(),
            shortage_penalty=problem.shortage_penalty,
            storage_capacity=problem.storage_capacity,
        )
        scenario_results, _ = solve_kitting_mip(modified_problem)

        total_demand = float(np.sum(modified_problem.demand))
        total_shortage = float(np.sum(scenario_results["shortages"]))
        service_level = 100 * (total_demand - total_shortage) / total_demand
        results_comparison.append(
            {
                "capacity_factor": scenario,
                "total_cost": scenario_results["objective"],
                "service_level": service_level,
                "total_production": float(
                    np.sum(scenario_results["production_schedule"])
                ),
            }
        )

    print("\nCapacity Scenario Comparison:")
    print("-" * 68)
    print(
        f"{'Capacity factor':<18} {'Total cost':<15} "
        f"{'Service level':<17} {'Total production':<18}"
    )
    print("-" * 68)
    for result in results_comparison:
        print(
            f"{result['capacity_factor']:<18.1f} "
            f"${result['total_cost']:<14.2f} "
            f"{result['service_level']:<16.1f}% "
            f"{result['total_production']:<18.1f}"
        )

    if len(results_comparison) > 1:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        capacities = [item["capacity_factor"] for item in results_comparison]
        costs = [item["total_cost"] for item in results_comparison]
        service_levels = [item["service_level"] for item in results_comparison]

        ax1.plot(capacities, costs, "bo-", linewidth=2, markersize=8)
        ax1.set_title("Optimized Cost vs. Capacity Factor")
        ax1.set_xlabel("Capacity factor relative to the base case")
        ax1.set_ylabel("Total cost (dollars)")
        ax1.set_xticks(capacities)
        ax1.grid(True, alpha=0.3)

        ax2.plot(capacities, service_levels, "go-", linewidth=2, markersize=8)
        ax2.set_title("Service Level vs. Capacity Factor")
        ax2.set_xlabel("Capacity factor relative to the base case")
        ax2.set_ylabel("Demand-weighted service level (%)")
        ax2.set_xticks(capacities)
        ax2.set_ylim(0, 100)
        ax2.grid(True, alpha=0.3)

        fig.tight_layout()
        plt.show()

    return results_comparison


sensitivity_results = sensitivity_analysis(problem)

## 6. Interpretation and limitations

The MILP provides an optimal schedule for the stated deterministic model when the solver reports an optimal status. Its objective makes explicit the modeled trade-off: producing earlier may increase inventory holding cost, while producing too little can incur shortage penalties; production must also fit within each shift's available minutes and component supply.

Interpret the reported service level together with the kit-by-kit shortages and the schedule. An aggregate service level can hide differences among kit types. Likewise, capacity utilization is a shift-period measure: high utilization indicates that little modeled capacity remains in that shift and period, but it does not by itself establish that additional capacity would reduce total cost.

The binary link variables do not create a setup charge in this example. Since the objective contains no term involving `y`, these variables only enforce that positive production is linked to an active binary variable; they do not represent a priced setup decision. Component supply is fixed at 500 units per SKU per period, component shortages are excluded, and storage capacity applies only to finished kits. These are deliberate scope assumptions, not claims about every kitting operation.

This mathematical formulation is useful when the planning data and constraints can be represented explicitly and an optimized benchmark is valuable. As the numbers of kits, SKUs, shifts, and periods grow, the MILP may require more computation; its practical performance depends on the formulation, data, and solver. Natural extensions include adding operational constraints or refining the input data while retaining the same modeling and MILP solution approach.